# Single-emitter observation-series r-GCN classification

This notebook trains a leakage-safe relational graph classifier on `generated/demo_esm_observation_series.json` for the five diagnostic targets used in `observation_etl_rgcn_end_to_end.ipynb`:

- `aircraft_type`
- `aircraft_variant`
- `radar_mode`
- `radar_type`
- `operator_country`

It follows the series concepts in `docs/single_emitter_observation_series.md`: each time-ordered observation remains an individual node, observations are linked by temporal continuity, mode-segment structure is represented without exposing labels as model inputs, and labels are used only as supervised targets/evaluation values.


In [ ]:
from __future__ import annotations

import json, math, random, time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "generated" / "demo_esm_observation_series.json"
ARTIFACT_DIR = ROOT / "artifacts" / "observation_series_rgcn_classification"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(DATA_PATH)
print(ARTIFACT_DIR)
print(DEVICE)


## Load series and create a label-free inference view

Ground-truth fields are retained in a separate target table, but are recursively stripped from the feature payload before graph construction. This prevents leakage from `ground_truth_label`, `ground_truth_track_label`, and `ground_truth_mode_sequence` into node features or edges.


In [ ]:
LEAKAGE_KEYS = {"ground_truth_label", "ground_truth_track_label", "ground_truth_mode_sequence"}
TARGETS = ["aircraft_type", "aircraft_variant", "radar_mode", "radar_type", "operator_country"]

def strip_ground_truth(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {k: strip_ground_truth(v) for k, v in obj.items() if k not in LEAKAGE_KEYS}
    if isinstance(obj, list):
        return [strip_ground_truth(v) for v in obj]
    return obj

raw = json.loads(DATA_PATH.read_text(encoding="utf-8"))
series_records = raw["observation_series"]
inference_records = strip_ground_truth(series_records)

# Targets are extracted only from the original payload and never merged back into features.
target_rows = []
for s in series_records:
    for obs in s["observations"]:
        gt = obs["ground_truth_label"]
        target_rows.append({
            "observation_id": obs["observation_id"],
            "series_id": obs["series_id"],
            "sequence_index": obs["sequence_index"],
            "aircraft_type": gt.get("aircraft_family") or gt.get("aircraft_type"),
            "aircraft_variant": gt.get("aircraft_variant"),
            "radar_mode": gt.get("mode") or gt.get("radar_mode"),
            "radar_type": gt.get("radar") or gt.get("radar_type"),
            "operator_country": gt.get("operator") or gt.get("operator_country"),
        })

serialized_features = json.dumps(inference_records)
assert "ground_truth" not in serialized_features, "Ground-truth fields leaked into inference view"
print(f"series={len(series_records):,}, observations={len(target_rows):,}")
print(target_rows[0])


## Feature engineering from observations, candidates, temporal context, and segments

Features use measured ESM parameters, approximate kinematics, elapsed time, sensor metadata, and candidate-list aggregate statistics. Candidate values are treated as ambiguous retrieval/candidate evidence, not labels. The segment index below is derived by comparing adjacent candidate radar names and radar-parameter changes in the label-free inference view, so it can mark possible mode transitions without reading true modes.


In [ ]:
def flatten_numeric(prefix: str, value: Any, out: dict[str, float]) -> None:
    if isinstance(value, dict):
        for k, v in value.items(): flatten_numeric(f"{prefix}.{k}" if prefix else k, v, out)
    elif isinstance(value, (int, float)) and not isinstance(value, bool):
        out[prefix] = float(value)

def top_candidate_key(obs: dict[str, Any], key: str) -> str | None:
    cands = obs.get("candidate_labels_from_shared_kg_features") or []
    return cands[0].get(key) if cands else None

def segment_indices(observations: list[dict[str, Any]]) -> list[int]:
    segs, current = [], 0
    prev_radar, prev_freq = None, None
    for obs in observations:
        radar = top_candidate_key(obs, "radar")
        freq = obs.get("esm_radar_parameters", {}).get("frequency_mhz")
        shift = False
        if prev_radar is not None and radar is not None and radar != prev_radar:
            shift = True
        if prev_freq is not None and freq is not None and abs(freq - prev_freq) > 750:
            shift = True
        if segs and shift:
            current += 1
        segs.append(current)
        prev_radar, prev_freq = radar, freq
    return segs

feature_rows, node_meta = [], []
for series in inference_records:
    obs_list = sorted(series["observations"], key=lambda o: o["sequence_index"])
    segs = segment_indices(obs_list)
    n = max(len(obs_list), 1)
    for obs, seg in zip(obs_list, segs):
        feats = {
            "elapsed_time_s": float(obs.get("elapsed_time_s", 0.0)),
            "sequence_fraction": float(obs.get("sequence_index", 0)) / max(n - 1, 1),
            "segment_index": float(seg),
            "candidate_count": float(len(obs.get("candidate_labels_from_shared_kg_features") or [])),
            "duration_s": float(series.get("duration_s", 0.0)),
            "observation_count": float(series.get("observation_count", n)),
        }
        flatten_numeric("esm", obs.get("esm_radar_parameters", {}), feats)
        flatten_numeric("kin", obs.get("approximate_kinematics", {}), feats)
        flatten_numeric("loc", obs.get("estimated_emitter_location", {}), feats)
        feature_rows.append(feats)
        node_meta.append({"observation_id": obs["observation_id"], "series_id": obs["series_id"], "sequence_index": obs["sequence_index"]})

feature_names = sorted({k for row in feature_rows for k in row})
X_np = np.asarray([[row.get(k, 0.0) for k in feature_names] for row in feature_rows], dtype=np.float32)
mu, sigma = X_np.mean(axis=0), X_np.std(axis=0)
sigma[sigma == 0] = 1.0
X = torch.tensor((X_np - mu) / sigma, dtype=torch.float32, device=DEVICE)
print(X.shape, feature_names[:10])


## Build a relational graph

The graph contains observation nodes and relation types inspired by the series design note: temporal adjacency (`NEXT_OBSERVATION` / `PREV_OBSERVATION`), same-emitter continuity (`SAME_EMITTER`), inferred same-segment links (`SAME_MODE_SEGMENT`), inferred mode-shift boundaries (`POSSIBLE_MODE_SHIFT`), and self loops. No relation is built from target labels.


In [ ]:
RELATIONS = {"self": 0, "next_observation": 1, "prev_observation": 2, "same_emitter": 3, "same_mode_segment": 4, "possible_mode_shift": 5}
edge_src, edge_dst, edge_type = [], [], []

def add_edge(i, j, rel):
    edge_src.append(i); edge_dst.append(j); edge_type.append(RELATIONS[rel])

for i in range(len(node_meta)): add_edge(i, i, "self")

series_to_indices = defaultdict(list)
for i, meta in enumerate(node_meta): series_to_indices[meta["series_id"]].append(i)

segment_by_node = {i: int(feature_rows[i]["segment_index"]) for i in range(len(feature_rows))}
for indices in series_to_indices.values():
    indices = sorted(indices, key=lambda i: node_meta[i]["sequence_index"])
    for a, b in zip(indices, indices[1:]):
        add_edge(a, b, "next_observation"); add_edge(b, a, "prev_observation")
        if segment_by_node[a] == segment_by_node[b]:
            add_edge(a, b, "same_mode_segment"); add_edge(b, a, "same_mode_segment")
        else:
            add_edge(a, b, "possible_mode_shift"); add_edge(b, a, "possible_mode_shift")
    # Sparse same-emitter reinforcement edges between observations two samples apart.
    for a, b in zip(indices, indices[2:]):
        add_edge(a, b, "same_emitter"); add_edge(b, a, "same_emitter")

edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long, device=DEVICE)
edge_types = torch.tensor(edge_type, dtype=torch.long, device=DEVICE)
print({name: int((edge_types == rid).sum().cpu()) for name, rid in RELATIONS.items()})


## Encode labels and create the required 0.5 / 0.3 / 0.2 train-test-validation split

Splitting is performed by `series_id`, not by individual observation, so adjacent observations from the same emitter track cannot cross from training into evaluation splits.


In [ ]:
label_vocab, y = {}, {}
for task in TARGETS:
    vals = [row[task] for row in target_rows]
    classes = sorted(set(vals))
    label_vocab[task] = classes
    idx = {v: i for i, v in enumerate(classes)}
    y[task] = torch.tensor([idx[v] for v in vals], dtype=torch.long, device=DEVICE)
    print(task, len(classes), Counter(vals).most_common(3))

series_ids = sorted(series_to_indices)
rng = np.random.default_rng(SEED)
rng.shuffle(series_ids)
n = len(series_ids)
n_train = int(round(0.5 * n))
n_test = int(round(0.3 * n))
split_series = {
    "train": set(series_ids[:n_train]),
    "test": set(series_ids[n_train:n_train+n_test]),
    "val": set(series_ids[n_train+n_test:]),
}
splits = {
    name: torch.tensor([i for i, m in enumerate(node_meta) if m["series_id"] in ids], dtype=torch.long, device=DEVICE)
    for name, ids in split_series.items()
}
print({k: len(v) for k, v in splits.items()})
print({k: len(v) for k, v in split_series.items()})


## Define a lightweight r-GCN with task-specific classification heads


In [ ]:
class RelGraphConv(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, num_relations: int, dropout: float = 0.1):
        super().__init__()
        self.rel_weights = nn.Parameter(torch.empty(num_relations, in_dim, out_dim))
        self.bias = nn.Parameter(torch.zeros(out_dim))
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.rel_weights)
    def forward(self, x, edge_index, edge_types):
        src, dst = edge_index
        out = torch.zeros(x.size(0), self.rel_weights.size(-1), device=x.device)
        deg = torch.zeros(x.size(0), device=x.device)
        for rel in range(self.rel_weights.size(0)):
            mask = edge_types == rel
            if not torch.any(mask): continue
            msg = x[src[mask]] @ self.rel_weights[rel]
            out.index_add_(0, dst[mask], msg)
            deg.index_add_(0, dst[mask], torch.ones(mask.sum(), device=x.device))
        out = out / deg.clamp_min(1).unsqueeze(-1)
        return self.dropout(F.relu(out + self.bias))

class SeriesRGCNClassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_relations, class_sizes, dropout=0.15):
        super().__init__()
        self.conv1 = RelGraphConv(in_dim, hidden_dim, num_relations, dropout)
        self.conv2 = RelGraphConv(hidden_dim, hidden_dim, num_relations, dropout)
        self.heads = nn.ModuleDict({task: nn.Linear(hidden_dim, size) for task, size in class_sizes.items()})
    def forward(self, x, edge_index, edge_types):
        h = self.conv1(x, edge_index, edge_types)
        h = self.conv2(h, edge_index, edge_types)
        return {task: head(h) for task, head in self.heads.items()}

model = SeriesRGCNClassifier(X.size(1), 96, len(RELATIONS), {t: len(label_vocab[t]) for t in TARGETS}).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
print(model)


## Train with checkpoints, per-epoch diagnostics, and TensorBoard logs


In [ ]:
try:
    from torch.utils.tensorboard import SummaryWriter
except Exception:
    SummaryWriter = None

def classification_loss(logits, indices):
    return sum(F.cross_entropy(logits[task][indices], y[task][indices]) for task in TARGETS)

def accuracy(logits, labels):
    return float((logits.argmax(dim=-1) == labels).float().mean().detach().cpu())

@torch.no_grad()
def split_metrics(split):
    model.eval(); idx = splits[split]; logits = model(X, edge_index, edge_types)
    metrics = {"loss": float(classification_loss(logits, idx).detach().cpu())}
    for task in TARGETS:
        metrics[f"{task}_acc"] = accuracy(logits[task][idx], y[task][idx])
    return metrics

EPOCHS = 75
ckpt_path = ARTIFACT_DIR / "best_model.pt"
writer = SummaryWriter(str(ARTIFACT_DIR / "tensorboard")) if SummaryWriter else None
history, best_val = [], math.inf

for epoch in range(1, EPOCHS + 1):
    model.train(); optimizer.zero_grad(set_to_none=True)
    logits = model(X, edge_index, edge_types)
    loss = classification_loss(logits, splits["train"])
    loss.backward(); optimizer.step()
    row = {"epoch": epoch}
    for split in ("train", "test", "val"):
        m = split_metrics(split)
        row.update({f"{split}_{k}": v for k, v in m.items()})
        if writer:
            writer.add_scalar(f"loss/{split}", m["loss"], epoch)
            for task in TARGETS: writer.add_scalar(f"accuracy/{split}_{task}", m[f"{task}_acc"], epoch)
    history.append(row)
    if row["val_loss"] < best_val:
        best_val = row["val_loss"]
        torch.save({"epoch": epoch, "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(), "label_vocab": label_vocab, "feature_names": feature_names, "splits": {k: v.detach().cpu().tolist() for k, v in splits.items()}, "history": history}, ckpt_path)
    diag = [f"epoch={epoch:04d}", f"train_loss={row['train_loss']:.4f}", f"test_loss={row['test_loss']:.4f}", f"val_loss={row['val_loss']:.4f}"]
    diag += [f"train_{task}_acc={row[f'train_{task}_acc']:.3f}" for task in TARGETS]
    diag += [f"test_{task}_acc={row[f'test_{task}_acc']:.3f}" for task in TARGETS]
    print(" | ".join(diag))

if writer: writer.close()
print(f"best checkpoint: {ckpt_path} (val_loss={best_val:.4f})")


## Final evaluation, reports, predictions, and plots


In [ ]:
checkpoint = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
final_metrics = {split: split_metrics(split) for split in ("train", "test", "val")}

model.eval()
with torch.no_grad():
    final_logits = model(X, edge_index, edge_types)
    probs = {task: F.softmax(final_logits[task], dim=-1).detach().cpu().numpy() for task in TARGETS}

predictions = []
for i, meta in enumerate(node_meta):
    rec = dict(meta)
    for task in TARGETS:
        pred_idx = int(probs[task][i].argmax())
        rec[f"true_{task}"] = target_rows[i][task]
        rec[f"pred_{task}"] = label_vocab[task][pred_idx]
        rec[f"pred_{task}_confidence"] = float(probs[task][i][pred_idx])
    predictions.append(rec)

summary = {"final_metrics": final_metrics, "best_epoch": int(checkpoint["epoch"]), "split_sizes": {k: int(v.numel()) for k, v in splits.items()}, "split_series_counts": {k: len(v) for k, v in split_series.items()}, "targets": TARGETS, "relations": RELATIONS, "leakage_guard": "ground truth keys stripped before feature and graph construction"}
(ARTIFACT_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "predictions.json").write_text(json.dumps(predictions, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))


In [ ]:
epochs = [r["epoch"] for r in history]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for split in ("train", "test", "val"):
    axes[0].plot(epochs, [r[f"{split}_loss"] for r in history], label=split)
axes[0].set_title("Classification loss by split")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("sum cross-entropy"); axes[0].legend(); axes[0].grid(True, alpha=.3)
for task in TARGETS:
    axes[1].plot(epochs, [r[f"test_{task}_acc"] for r in history], label=f"test {task}")
axes[1].set_title("Test accuracy by task")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=.3)
fig.tight_layout()
plot_path = ARTIFACT_DIR / "training_metrics.png"
fig.savefig(plot_path, dpi=150)
plot_path
